# DeBERTa-based PCL Binary Classification

This notebook implements a DeBERTa model for Patronizing and Condescending Language (PCL) binary classification.

In [16]:
import os
import numpy as np
import pandas as pd
import torch
import random
from tqdm import tqdm
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, classification_report

# Set seed for reproducibility
seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [17]:
# Hyperparameters
max_length = 256
batch_size = 16
epochs = 5
learning_rate = 1e-5
model_name = "microsoft/deberta-base"
model_output_dir = "models"
patience = 3

# Augmentation parameters
augment_data = False
num_paraphrases_per_example = 2  # Number of paraphrases to generate per positive example

In [18]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Changed from 'label' to 'labels' for Trainer
        }

In [19]:
data = pd.read_csv(
    "data/dontpatronizeme_pcl.tsv",
    sep='\t',
    header=None,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"]
)

# Filter out missing text and invalid labels
data = data.dropna(subset=['text', 'label'])

# Convert to binary classification: PCL (>1) vs non-PCL (<=1)
data['binary_label'] = (data['label'] > 1).astype(int)

train_ids = pd.read_csv("data/train_semeval_parids-labels.csv")
dev_ids = pd.read_csv("data/dev_semeval_parids-labels.csv")

train_data = data[data['par_id'].isin(train_ids['par_id'])].copy()
dev_data = data[data['par_id'].isin(dev_ids['par_id'])].copy()

print(f"Train size (before augmentation): {len(train_data)}")
print(f"Dev size: {len(dev_data)}")
print(f"\nBinary label distribution (0=non-PCL, 1=PCL):")
print(train_data['binary_label'].value_counts())
print(f"\nClass ratio: {train_data['binary_label'].value_counts()[0] / train_data['binary_label'].value_counts()[1]:.1f}:1")

Train size (before augmentation): 8375
Dev size: 2093

Binary label distribution (0=non-PCL, 1=PCL):
binary_label
0    7581
1     794
Name: count, dtype: int64

Class ratio: 9.5:1


In [20]:
# Data Augmentation: EDA (Easy Data Augmentation)
if augment_data:
    print("\n" + "=" * 50)
    print("Data Augmentation - EDA")
    print("=" * 50)
    

    # Download WordNet
    try:
        import nltk
        nltk.data.find('corpora/wordnet')
    except:
        nltk.download('wordnet', quiet=True)
    
    from nltk.corpus import wordnet
    
    def eda_augment(text, n_syn=2, n_swap=1):
        """Apply synonym replacement and random swap"""
        words = text.split()
        
        # Synonym replacement
        for _ in range(n_syn):
            if words:
                idx = random.randint(0, len(words)-1)
                syns = [l.name() for s in wordnet.synsets(words[idx]) for l in s.lemmas()]
                if syns:
                    words[idx] = random.choice(syns).replace('_', ' ')
        
        # Random swap
        if len(words) >= 2:
            for _ in range(n_swap):
                i, j = random.sample(range(len(words)), 2)
                words[i], words[j] = words[j], words[i]
        
        return ' '.join(words)
    
    # Augment positive examples
    positive_examples = train_data[train_data['binary_label'] == 1]
    print(f"\nGenerating {num_paraphrases_per_example} augmented examples for {len(positive_examples)} positive examples...")
    
    augmented_texts = []
    for _, row in tqdm(positive_examples.iterrows(), total=len(positive_examples), desc="Augmenting"):
        for _ in range(num_paraphrases_per_example):
            try:
                aug = eda_augment(row['text'])
                if aug != row['text']:
                    augmented_texts.append(aug)
            except:
                pass
    
    # Add augmented data
    augmented_df = pd.DataFrame({
        'par_id': range(len(train_data), len(train_data) + len(augmented_texts)),
        'text': augmented_texts,
        'binary_label': 1
    })
    
    original_size = len(train_data)
    train_data = pd.concat([train_data, augmented_df], ignore_index=True)
    
    print(f"\nOriginal: {original_size} | Added: {len(augmented_texts)} | New total: {len(train_data)}")
    print(f"New class ratio: {(train_data['binary_label']==0).sum() / (train_data['binary_label']==1).sum():.1f}:1")
    
else:
    print("\nData augmentation disabled")

# Load tokenizer and model
print("\n" + "=" * 50)
print("Loading Model")
print("=" * 50)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(f"Model: {model_name}")
print(f"Number of labels: {model.num_labels}")


Data augmentation disabled

Loading Model


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: microsoft/deberta-base
Number of labels: 2


In [21]:
train_data[train_data['binary_label'] == 1]

,par_id,art_id,keyword,country_code,text,label,binary_label
32,33,@@8301323,disabled,pk,Arshad said that besides learning many new asp...,2,1
33,34,@@24150149,disabled,ng,Fast food employee who fed disabled man become...,3,1
41,42,@@4591694,hopeless,jm,Vanessa had feelings of hopelessness in her fi...,3,1
76,77,@@22454828,homeless,nz,"In September , Major Nottle set off on foot fr...",3,1
82,83,@@4672144,homeless,pk,The demographics of Pakistan and India are ver...,3,1
...,...,...,...,...,...,...,...
10423,10424,@@4665292,women,jm,""" I do n't believe in abortion , I think it is...",3,1
10444,10445,@@3923193,refugee,gb,More than 150 volunteers spent the night in ' ...,3,1
10453,10454,@@22338535,vulnerable,ie,""" We are challenged , I suggest , to turn this...",4,1
10466,10467,@@20282330,in-need,ng,""" She has one huge platform , and information ...",3,1


In [22]:
# Create datasets
train_dataset = PCLDataset(
    train_data['text'].values,
    train_data['binary_label'].values,
    tokenizer,
    max_length=max_length
)

dev_dataset = PCLDataset(
    dev_data['text'].values,
    dev_data['binary_label'].values,
    tokenizer,
    max_length=max_length
)

# Calculate class weights for weighted sampling
train_labels_binary = train_data['binary_label'].values
k_p = np.mean(train_labels_binary == 1)
k_n = np.mean(train_labels_binary == 0)

sample_weights = np.zeros(len(train_labels_binary))
sample_weights[train_labels_binary == 1] = 1 / np.sqrt(k_p)
sample_weights[train_labels_binary == 0] = 1 / np.sqrt(k_n)

print(f"Class weights calculated: positive={1/np.sqrt(k_p):.3f}, negative={1/np.sqrt(k_n):.3f}")

Class weights calculated: positive=3.248, negative=1.051


In [23]:
# Define metric computation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='binary', pos_label=1)
    return {'f1': f1}

# Define custom Trainer with weighted sampling
from torch.utils.data import WeightedRandomSampler

class WeightedTrainer(Trainer):
    def __init__(self, sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.sample_weights = sample_weights
    
    def _get_train_sampler(self, dataset):
        if self.sample_weights is not None:
            return WeightedRandomSampler(
                self.sample_weights,
                len(self.sample_weights),
                replacement=True
            )
        return super()._get_train_sampler(dataset)

# Training arguments
training_args = TrainingArguments(
    output_dir=model_output_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=learning_rate,
    weight_decay=0.01,
    warmup_steps=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=1,
)

# Initialize trainer
trainer = WeightedTrainer(
    sample_weights=sample_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)]
)

print("Trainer initialized with weighted sampling and early stopping")

Trainer initialized with weighted sampling and early stopping


In [24]:
# Train the model
trainer.train()

# Save the best model
trainer.save_model(os.path.join(model_output_dir, 'best_model'))
tokenizer.save_pretrained(os.path.join(model_output_dir, 'best_model'))

print(f"\nTraining complete! Best model saved to {model_output_dir}/best_model")

Epoch,Training Loss,Validation Loss,F1
1,0.419209,0.216944,0.583333
2,0.193403,0.293157,0.621687
3,0.121935,0.364950,0.594848
4,0.072968,0.439182,0.551515
5,0.034506,0.421718,0.587912


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete! Best model saved to models/best_model


In [37]:
print("\n" + "=" * 50)
print("Final Evaluation")
print("=" * 50)

# Get predictions on dev set
predictions = trainer.predict(dev_dataset)
final_preds = np.argmax(predictions.predictions, axis=-1)
final_probs = torch.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()
final_labels = predictions.label_ids


Final Evaluation


In [40]:
# Classification report
print("\nClassification Report:")
print(classification_report(final_labels, final_preds, target_names=['non-PCL', 'PCL']))

# Per-class F1
print(f"\nF1 (binary, pos_label=1): {f1_score(final_labels, final_preds, average='binary', pos_label=1):.4f}")



Classification Report:
              precision    recall  f1-score   support

     non-PCL       0.96      0.95      0.96      1894
         PCL       0.60      0.64      0.62       199

    accuracy                           0.92      2093
   macro avg       0.78      0.80      0.79      2093
weighted avg       0.93      0.92      0.93      2093


F1 (binary, pos_label=1): 0.6184


In [41]:
# Generate predictions for submission
# Save dev predictions
print("Saving dev.txt...")
np.savetxt('dev.txt', final_preds, fmt='%d')
print(f"Dev predictions saved: {len(final_preds)} lines")

# Load test data and generate predictions
print("\nLoading test data...")
test_data = pd.read_csv("data/task4_test.tsv", sep='\t', header=None, 
                        names=["par_id", "art_id", "keyword", "country_code", "text"])
print(f"Test size: {len(test_data)}")

# Create test dataset (using dummy labels)
test_dataset = PCLDataset(
    test_data['text'].values,
    np.zeros(len(test_data)),  # dummy labels
    tokenizer,
    max_length=max_length
)

# Get predictions on test set
print("Generating test predictions...")
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)

# Save test predictions
np.savetxt('test.txt', test_preds, fmt='%d')
print(f"Test predictions saved: {len(test_preds)} lines")

# Verify files
print("\nVerification:")
print(f"dev.txt lines: {len(open('dev.txt').readlines())}")
print(f"test.txt lines: {len(open('test.txt').readlines())}")

Saving dev.txt...
Dev predictions saved: 2093 lines

Loading test data...
Test size: 3832
Generating test predictions...


Test predictions saved: 3832 lines

Verification:
dev.txt lines: 2093
test.txt lines: 3832
